In [1]:
import os
from google import genai
from PIL import Image, ImageOps
from google.genai import types
from pydantic import BaseModel, Field
from typing import Optional
import io
import pandas as pd
from dotenv import load_dotenv

In [15]:
current_folder = os.getcwd()
image_path = os.path.join(current_folder, "raw data", "photo_8.jpg")
img = Image.open(image_path)
img = img.transpose(Image.ROTATE_90)
#img = ImageOps.exif_transpose(img)
img.show()

In [6]:
current_folder = os.getcwd()
print(current_folder)

c:\Users\Stephen Williams\Downloads\EDA ORTHOpedic


In [16]:
expected_rows = 13
prompt = f"""You are a specialised medical document extractor. Your role is to extract structured information from this image. In this image, there is a handwritten table with 7 columns. The features are Date, year, region, occupation, M/F, Age, diagnosis.
It has {expected_rows} rows. I want you to extract the information from this table and return it in a strcutured format like a nest list. Ignore the header
Rules to follow:
1. The output should not include the header row of the table.
2. Track horizontal rows carefully: Ensure the diagnosis text matches the exact horizontal row line of the Age and Date columns. Do not let text shift down into the next row's list.
3. Capture multi-line cells: In the Diagnosis column, a single cell often has text written on two or three stacked lines (e.g., "Chondromalasia Grade I" with "Mild OA." written directly beneath it). You MUST capture all lines within that row's cell and combine them into a single string.
4. The output should be in a format that can be easily converted to a pandas dataframe.
5. Extract the information as accurately as possible. For example Chondromalacia is written as chondromalasia, the s written instead of the c. So extract it as chondromalasia.
6. If there are any missing values in the table, represent them as an empty string (" ") in the output list.
7. respect abreviations.right maybe be written as rt
8. Sometimes the rows aren't separated by a line. So you need to track the horizontal position of the text to determine which row it belongs to. Do not rely on line separation alone.You can assume that if the Diagnosis text is on the same horizontal line as the Date, Age and M/F columns, then it belongs to that row. If the Diagnosis text is on a different horizontal line, then it belongs to a different row.
"""



In [6]:
class TableRow(BaseModel):
    date: str = Field(description="The date of the record")
    year: Optional[str] = Field(description="The year of the record. you will see alot of empty values in this column so if there is an empty value, just put an empty string")
    region: str = Field(description="The region associated with the record. Its just a number but it represents a region")
    occupation: str = Field(description="The occupation of the patient")
    gender: str = Field(description="The gender of the patient. It can be M or F")
    age: Optional[str] = Field(description="The age of the patient.")
    diagnosis: str = Field(description="The diagnosis text of the patient")

In [ ]:
from google import genai
from google.genai import types
import io

img_byte_arr = io.BytesIO()
img.save(img_byte_arr, format='JPEG')
img_bytes = img_byte_arr.getvalue()

image_part = types.Part.from_bytes(data=img_bytes, mime_type="image/jpeg")
load_dotenv()

client = genai.Client()

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=[image_part, prompt],
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=list[TableRow],
        temperature=0.0,
),
)

print(response.parsed)

In [ ]:

cleaned_rows = []

for row in response.parsed:
    
  
    row_dict = row.model_dump()
    
   
    for key, value in row_dict.items():
        if isinstance(value, str):
            
            row_dict[key] = value.replace('\n', ' ').strip()
            

    cleaned_rows.append(row_dict)


df = pd.DataFrame(cleaned_rows)

#
df.to_csv("orthopedic_data_clean_8.csv", index=False)

# 5. Preview your new structured datasetprint("Pipeline complete! Here is a sneak peek of your data:")
print(df.head())

In [ ]:
from IPython.display import display
display(df.head())